# Evaluating Agents on the Agent Platform — Companion Notebook

**A hands-on companion to the deck [`docs/eval_slides.pptx`](eval_slides.pptx) / [`eval_slides.html`](eval_slides.html).**

The slides tell the story; this notebook lets you *run* it. Each section maps 1:1 to a slide,
pairs the console screenshot with runnable code, and walks the **Quality Flywheel**:

> **Design → Execution → Scoring → Refinement → (repeat)**

Everything here covers Google's *Gemini Enterprise Agent Platform → Optimize → Evaluation* docs.
For the full operations reference and the doc-coverage matrix, see
[`docs/eval_operations.md`](eval_operations.md) and the guided walkthrough
[`docs/evaluation_demo.md`](evaluation_demo.md).

---
### How to use this notebook
- It runs **offline by default** — no cloud credentials required. Every step degrades gracefully to a
  demonstration/fallback so the notebook always executes top-to-bottom with output.
- To run **live** against your deployed agent, authenticate (`gcloud auth application-default login`),
  set `AGENT_ENGINE_ID`, and use the live orchestrator shown in the last section.
- It reuses the exact same code the CLI demo runs (`src/eval/demo/…`), so what you learn here is what
  ships.

## Setup
Locate the repo, import the shared demo code, and print the environment. This cell never fails — if the modules aren't importable in your kernel it flips to *doc-only* mode and the rest of the notebook still renders.

In [1]:
import os, sys

# Find the repo root (dir containing src/config.py) from wherever the kernel started.
ROOT = os.getcwd()
for _ in range(8):
    if os.path.isfile(os.path.join(ROOT, "src", "config.py")):
        break
    ROOT = os.path.dirname(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)   # so any relative demo writes land at the repo root, not next to this notebook
os.environ.setdefault("EVAL_OUTPUT_DIR", os.path.join(ROOT, "eval_outputs", "companion"))

try:
    from IPython.display import Markdown, display
    def md(s): display(Markdown(s))
except Exception:                       # pragma: no cover - non-IPython fallback
    def md(s): print(s)

DEMO_AVAILABLE, IMPORT_ERROR = True, ""
try:
    from src.config import AGENT_ENGINE_ID
    from src.eval.demo import steps
    from src.eval.demo.full_eval_demo import run_demo
    RESOURCE = steps.resolve_resource(AGENT_ENGINE_ID)
except Exception as e:                   # bare env / missing deps -> doc-only mode
    DEMO_AVAILABLE, IMPORT_ERROR = False, f"{type(e).__name__}: {e}"

print("repo root           :", ROOT)
print("demo code importable:", DEMO_AVAILABLE, IMPORT_ERROR)
if DEMO_AVAILABLE:
    print("AGENT_ENGINE_ID     :", AGENT_ENGINE_ID)
    print("agent resource      :", RESOURCE)
    print("\nRunning OFFLINE (no credentials needed). See the last section to run live.")

repo root           : /home/admin_jwortz_altostrat_com/geap-tour/.claude/worktrees/notebook
demo code importable: True 
AGENT_ENGINE_ID     : 5895016748914049024
agent resource      : projects/wortz-project-352116/locations/us-central1/reasoningEngines/5895016748914049024

Running OFFLINE (no credentials needed). See the last section to run live.


## 1 · The evaluation loop  <sub>(slide 2)</sub>

Turn *"it seems to work"* into something you can **measure**. Evaluation is a loop across four phases,
and the platform gives you a place to do each one:

| Phase | Question it answers | Platform surface |
|-------|---------------------|------------------|
| **Design** | *What does "good" mean here?* | Metric Registry (predefined + custom) |
| **Execution** | *How does the agent actually behave?* | Rapid / test-case / simulated / offline runs → traces |
| **Scoring** | *Is quality holding up in production?* | Online Monitors → Cloud Monitoring |
| **Refinement** | *Why did it fail, and how do we fix it?* | Trace/BigQuery analysis → prompt optimization |

![Agent traces in the console](screenshots/eval_console_agent_traces.png)

The cell below runs the **entire flywheel offline** and summarizes each step.

In [2]:
if DEMO_AVAILABLE:
    report = run_demo(offline=True)
    rows = "\n".join(
        f"| {s['step']} | {s['title']} | `{s['status']}` | [{s['doc'].rsplit('/',1)[-1]}]({s['doc']}) |"
        for s in report["steps"]
    )
    md(f"### Flywheel run — {report['steps_ok']}/{report['steps_total']} steps produced output "
       f"(mode: `{report['mode']}`)\n\n"
       f"| # | Step | Status | Doc page |\n|--:|------|--------|----------|\n{rows}")
else:
    md(f"> _Doc-only mode ({IMPORT_ERROR}). The code below is shown for reading; run it inside the "
       f"geap-tour environment to execute._")

  GEAP EVALUATION DEMO — 100% coverage of Optimize > Evaluation  [OFFLINE (fixtures/fallbacks)]

----- Phase 1: Design (define metrics) -----

>>> [manage-metrics]


    step 1 — Metric Registry (predefined + custom LLM + custom code + exact-match) :: OK

----- Phase 2: Execution (run inferences, generate traces) -----
    (offline: skipping live rapid/testcase/simulate inference steps)

>>> [evaluate-simulated/env]
Environment simulation demo — agent_name='travel_agent'
In production, replace each MCP toolset on the agent with an
intercepted version, then run the simulated-user eval:
    live_tools = {t.name: t for t in agent.tools}         # MCP tools
    wrapped    = wrap_tools(live_tools, mocks=..., error_every=3)
    # ...attach `wrapped` to the agent, then:
    run_simulated_eval(agent_resource, agent_name=..., multi_turn=True)

Wrapped 3 tools: ['check_expense_policy', 'search_flights', 'search_hotels']
Error injection: every 3 call(s)
------------------------------------------------------------------------
  call 1: search_flights -> mocked data: [{'flight_id': 'FL001', 'route': 'SFO->JFK', 'price': 420, 'airline': 'United'}, {'flight_id': 

  BigQuery unavailable (400 GET https://bigquery.googleapis.com/bigquery/v2/projects/wortz-project-352116/queries/d806aa8f-84e8-4d9b-8e7c-997afbabe9be?maxResults=0&location=US&prettyPrint=false: Unrecognized name: json_payload at [11:22]

Location: US
Job ID: d806aa8f-84e8-4d9b-8e7c-997afbabe9be
); falling back to bundled fixture.
  Source: fixture /home/admin_jwortz_altostrat_com/geap-tour/.claude/worktrees/notebook/src/eval/sample_traces.jsonl (8 traces)
  Scoring 8 historical traces with 3 metrics...


/home/admin_jwortz_altostrat_com/geap-tour/.claude/worktrees/notebook/src/eval/offline_trace_eval.py:283: FutureWarning: The vertexai.Client class is deprecated. Please use agentplatform.Client instead.
  client = Client(project=GCP_PROJECT_ID, location=GCP_REGION)


17:33:01 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'


...

Computing Metrics for Evaluation Dataset: 100%|██████████| 24/24 [03:15<00:00, 43.12s/it]

Computing Metrics for Evaluation Dataset: 100%|██████████| 24/24 [03:15<00:00,  8.16s/it]


  Per-metric results:
    [FAIL] final_response_quality_v1: score=2.40/5 (raw=0.479, total=8, errors=0, threshold=3.0)
    [PASS] hallucination_v1: score=5.00/5 (raw=1.0, total=8, errors=0, threshold=3.0)
    [PASS] safety_v1: score=4.38/5 (raw=0.875, total=8, errors=0, threshold=3.0)
    step 6 — Offline evaluation over historical traces/sessions :: OK

----- Phase 3: Scoring (production monitoring) -----

>>> [evaluate-online]
    step 7 — Continuous evaluation with Online Monitors :: OK

----- Phase 4: Refinement (optimize) -----

>>> [optimize-agent]
[sdk_optimize] client.optimizer is unavailable in this aiplatform SDK version (the documented client.optimizer.optimize(...) path does not exist here). Falling back to the ADK GEPA optimizer (src/optimize/run_optimize.py). See: https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/optimize-agent
    step 9 — Optimize agent prompts (Quality Flywheel) :: OK

>>> [quality-alerts]


✓ Alert policy YAML written: src/eval/policies/quality_drift_policy.yaml
  Metric: aiplatform.googleapis.com/online_evaluator/scores (evaluation_metric_name=task_success) < 0.8
  Apply: gcloud monitoring policies create --policy-from-file=src/eval/policies/quality_drift_policy.yaml
    step 10 — Quality-drift alerts (Cloud Monitoring policy) :: OK

  DEMO COMPLETE — 6/6 steps ran live; 0 skipped (offline/no-cred).


### Flywheel run — 6/6 steps produced output (mode: `offline`)

| # | Step | Status | Doc page |
|--:|------|--------|----------|
| 1 | Metric Registry (predefined + custom LLM + custom code + exact-match) | `ok` | [manage-metrics](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/manage-metrics) |
| 5 | Environment simulation (mock tools + injected 503s) | `ok` | [evaluate-simulated](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-simulated) |
| 6 | Offline evaluation over historical traces/sessions | `ok` | [evaluate-offline](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-offline) |
| 7 | Continuous evaluation with Online Monitors | `ok` | [evaluate-online](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-online) |
| 9 | Optimize agent prompts (Quality Flywheel) | `ok` | [optimize-agent](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/optimize-agent) |
| 10 | Quality-drift alerts (Cloud Monitoring policy) | `ok` | [quality-alerts](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/quality-alerts) |

## 2 · Design — pick a metric, or write your own  <sub>(slide 3)</sub>

Define a metric **once** and reuse it across offline runs *and* online monitors. There are three kinds,
plus the split between **reference-based** (needs a golden answer) and **reference-free** (judged on its
own merits):

1. **Predefined rubric metrics** — e.g. `FINAL_RESPONSE_QUALITY`, `SAFETY`, `TOOL_USE_QUALITY`, and the
   multi-turn variants (`MULTI_TURN_TASK_SUCCESS`, …). LLM-as-judge, reference-free.
2. **Custom LLM-as-judge** — your own rubric via `MetricPromptBuilder`.
3. **Custom code metric** — deterministic Python (`CodeExecutionMetric`), e.g. a hard policy-limit check.
   Plus a reference-based **Exact Match** for regression-style checks.

![Metrics tab in the console](screenshots/eval_console_metrics_tab.png)

In [3]:
if DEMO_AVAILABLE:
    out = steps.register_metrics(client=None, do_register=False)   # build the catalog (no cloud write)
    cat = out.get("catalog", {})
    md("**Metric Registry catalog** (from `src/eval/metric_registry.py`)\n\n"
       f"- Predefined **single-turn** rubric: `{cat.get('single_turn_rubric')}`\n"
       f"- Predefined **multi-turn** rubric: `{cat.get('multi_turn_rubric')}`\n"
       f"- **Custom** (LLM + code + exact-match): `{cat.get('custom')}`\n"
       f"- Reference-based Exact Match present: `{cat.get('exact_match_reference_based')}`\n"
       f"- Custom code metric present: `{cat.get('code_metric')}`\n\n"
       "> To register these in the cloud Metric Registry: `steps.register_metrics(client, do_register=True)`.")
else:
    print("doc-only mode")

**Metric Registry catalog** (from `src/eval/metric_registry.py`)

- Predefined **single-turn** rubric: `['FINAL_RESPONSE_QUALITY', 'HALLUCINATION', 'TOOL_USE_QUALITY', 'SAFETY']`
- Predefined **multi-turn** rubric: `['MULTI_TURN_TASK_SUCCESS', 'MULTI_TURN_TOOL_USE_QUALITY', 'MULTI_TURN_TRAJECTORY_QUALITY']`
- **Custom** (LLM + code + exact-match): `['policy_compliance', 'geap_tool_use', 'policy_limit_exact', 'exact_match']`
- Reference-based Exact Match present: `True`
- Custom code metric present: `True`

> To register these in the cloud Metric Registry: `steps.register_metrics(client, do_register=True)`.

## 3 · Execution — three ways to exercise the agent  <sub>(slide 4)</sub>

A **trace** is the record of one run: the model inputs, the responses, and the tool calls. Produce them
whichever way fits the stage:

- **Rapid eval** — a quick pointwise LLM-judge run on a handful of prompts (`client.evals.evaluate`).
- **Test-case / regression** — a batch suite against the deployed agent; fail the build on drift.
- **Simulated multi-turn** — auto-generate user scenarios and score full conversations.
- **Environment simulation** — mock the tools and inject errors (e.g. 503s) to test resilience.
- **Offline eval** — score historical traces/sessions with no re-inference.

![Agent Engines in the console](screenshots/eval_console_agent_engines.png)

The rapid / test-case / simulated runs need live inference; the cell below runs the two that work fully
**offline** — environment simulation and offline-trace scoring.

In [4]:
if DEMO_AVAILABLE:
    env = steps.environment_simulation()                    # mock tools + injected 503s (offline)
    off = steps.offline_eval(client=None)                   # score historical traces (offline fallback)
    md(f"- **Environment simulation** → `{env.get('status')}` — "
       f"`{env.get('summary', env.get('reason'))}`\n"
       f"- **Offline trace eval** → `{off.get('status')}` — "
       f"`{off.get('result', off.get('reason'))}`\n\n"
       "> Live inference variants (shown for reference — require credentials):\n"
       "> ```python\n"
       "> steps.rapid_eval(client, RESOURCE)                # client.evals.evaluate + result.show()\n"
       "> steps.testcase_eval(AGENT_ENGINE_ID, 'coordinator_agent')  # regression batch\n"
       "> steps.simulate(RESOURCE, 'coordinator_agent')     # simulated multi-turn\n"
       "> ```")
else:
    print("doc-only mode")

Environment simulation demo — agent_name='travel_agent'
In production, replace each MCP toolset on the agent with an
intercepted version, then run the simulated-user eval:
    live_tools = {t.name: t for t in agent.tools}         # MCP tools
    wrapped    = wrap_tools(live_tools, mocks=..., error_every=3)
    # ...attach `wrapped` to the agent, then:
    run_simulated_eval(agent_resource, agent_name=..., multi_turn=True)

Wrapped 3 tools: ['check_expense_policy', 'search_flights', 'search_hotels']
Error injection: every 3 call(s)
------------------------------------------------------------------------
  call 1: search_flights -> mocked data: [{'flight_id': 'FL001', 'route': 'SFO->JFK', 'price': 420, 'airline': 'United'}, {'flight_id': 'FL002', 'route': 'SFO->JFK', 'price': 510, 'airline': 'Delta'}]
  call 2: search_flights -> mocked data: [{'flight_id': 'FL001', 'route': 'SFO->JFK', 'price': 420, 'airline': 'United'}, {'flight_id': 'FL002', 'route': 'SFO->JFK', 'price': 510, 'airline'

  BigQuery unavailable (400 GET https://bigquery.googleapis.com/bigquery/v2/projects/wortz-project-352116/queries/b6ddc139-3303-4c99-98f6-066e7c6531b6?maxResults=0&location=US&prettyPrint=false: Unrecognized name: json_payload at [11:22]

Location: US
Job ID: b6ddc139-3303-4c99-98f6-066e7c6531b6
); falling back to bundled fixture.
  Source: fixture /home/admin_jwortz_altostrat_com/geap-tour/.claude/worktrees/notebook/src/eval/sample_traces.jsonl (8 traces)
  Scoring 8 historical traces with 3 metrics...


Computing Metrics for Evaluation Dataset:   0%|          | 0/24 [00:00<?, ?it/s]

Computing Metrics for Evaluation Dataset:   4%|▍         | 1/24 [00:03<01:19,  3.44s/it]

Computing Metrics for Evaluation Dataset:   8%|▊         | 2/24 [00:05<00:52,  2.37s/it]

...

Computing Metrics for Evaluation Dataset:  96%|█████████▌| 23/24 [00:45<00:02,  2.59s/it]

Computing Metrics for Evaluation Dataset: 100%|██████████| 24/24 [01:07<00:00,  8.36s/it]

Computing Metrics for Evaluation Dataset: 100%|██████████| 24/24 [01:07<00:00,  2.82s/it]


  Per-metric results:
    [FAIL] final_response_quality_v1: score=1.46/5 (raw=0.292, total=8, errors=0, threshold=3.0)
    [PASS] hallucination_v1: score=5.00/5 (raw=1.0, total=8, errors=0, threshold=3.0)
    [PASS] safety_v1: score=4.38/5 (raw=0.875, total=8, errors=0, threshold=3.0)


- **Environment simulation** → `ok` — `{'agent_name': 'travel_agent', 'agent_resource': None, 'tools_wrapped': ['check_expense_policy', 'search_flights', 'search_hotels'], 'total_calls': 6, 'mock_hits': 5, 'injected_errors': 1, 'error_every': 3}`
- **Offline trace eval** → `ok` — `{'status': 'ok', 'source': 'fixture', 'agent_name': 'coordinator_agent', 'hours_back': 24, 'record_count': 8, 'score_threshold': 3.0, 'timestamp': '2026-07-28T17:37:34.607407', 'metrics': {'final_response_quality_v1': {'raw_mean': 0.2916666675, 'score': 1.4583333374999998, 'num_cases_total': 8, 'num_cases_error': 0, 'status': 'FAIL'}, 'hallucination_v1': {'raw_mean': 1.0, 'score': 5.0, 'num_cases_total': 8, 'num_cases_error': 0, 'status': 'PASS'}, 'safety_v1': {'raw_mean': 0.875, 'score': 4.375, 'num_cases_total': 8, 'num_cases_error': 0, 'status': 'PASS'}}}`

> Live inference variants (shown for reference — require credentials):
> ```python
> steps.rapid_eval(client, RESOURCE)                # client.evals.evaluate + result.show()
> steps.testcase_eval(AGENT_ENGINE_ID, 'coordinator_agent')  # regression batch
> steps.simulate(RESOURCE, 'coordinator_agent')     # simulated multi-turn
> ```

What a **live** batch run looks like in the console (per-metric means and per-case scores):

<table><tr>
<td><img src="screenshots/eval_batch_scores.png" width="470"></td>
<td><img src="screenshots/eval_per_case_scores.png" width="470"></td>
</tr></table>

## 4 · Scoring — watch quality in production  <sub>(slide 5)</sub>

**Online Monitors** score live traffic as it arrives and send the scores to **Cloud Monitoring**, so you
catch quality drift *before* users do. The monitor runs an async loop — *Query → Evaluate → Report* —
roughly every 10 minutes and exports to Cloud Logging + Cloud Monitoring, where you chart it and alert on it.

![Online monitors](screenshots/eval_console_online_monitors.png)

The Metrics Explorer dashboard and a firing quality alert:

<table><tr>
<td><img src="screenshots/eval_console_metrics_dashboard.png" width="470"></td>
<td><img src="screenshots/eval_console_monitoring_alerts.png" width="470"></td>
</tr></table>

In [5]:
if DEMO_AVAILABLE:
    mon = steps.online_monitors(do_setup=False)             # describe the monitor loop (no cloud write)
    alr = steps.quality_alerts()                            # write the Cloud Monitoring alert policy YAML
    md(f"- **Online Monitors** → `{mon.get('status')}`\n\n  {mon.get('note','')}\n\n"
       f"- **Quality-drift alert policy** → `{alr.get('status')}` — "
       f"wrote `{alr.get('policy_file', alr.get('reason'))}`\n\n"
       "> Create/verify the monitors live with:\n"
       "> `python -m src.eval.setup_online_evaluators create|verify`")
else:
    print("doc-only mode")

✓ Alert policy YAML written: src/eval/policies/quality_drift_policy.yaml
  Metric: aiplatform.googleapis.com/online_evaluator/scores (evaluation_metric_name=task_success) < 0.8
  Apply: gcloud monitoring policies create --policy-from-file=src/eval/policies/quality_drift_policy.yaml


- **Online Monitors** → `ok`

  Online Monitors asynchronously score live traces on a ~10-min loop (Query -> Evaluate -> Report), exporting to Cloud Logging + Cloud Monitoring. Create/verify with: python -m src.eval.setup_online_evaluators create|verify

- **Quality-drift alert policy** → `ok` — wrote `src/eval/policies/quality_drift_policy.yaml`

> Create/verify the monitors live with:
> `python -m src.eval.setup_online_evaluators create|verify`

## 5 · Refinement — close the flywheel  <sub>(slide 6)</sub>

Don't just learn *that* the agent failed — learn **why**, then fix it. Route traces and scores to
BigQuery, cluster the failures, and feed the worst cases back into **prompt optimization** (the SDK
optimizer, with an ADK-GEPA fallback). That's the loop that keeps the number going up.

![Traces + scores in BigQuery](screenshots/eval_console_bigquery.png)

In [6]:
if DEMO_AVAILABLE:
    opt = steps.optimize(client=None)                       # optimizer path (ADK-GEPA fallback offline)
    r = opt.get("result", opt.get("reason"))
    md(f"- **Optimize** → `{opt.get('status')}`\n\n  `{str(r)[:600]}`")
else:
    print("doc-only mode")

[sdk_optimize] client.optimizer is unavailable in this aiplatform SDK version (the documented client.optimizer.optimize(...) path does not exist here). Falling back to the ADK GEPA optimizer (src/optimize/run_optimize.py). See: https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/optimize-agent


- **Optimize** → `ok`

  `{'method': 'adk_gepa_fallback', 'status': 'skipped', 'reason': "GEPA is a live 10-20 min optimization; set GEAP_RUN_GEPA=1 and run in an authenticated GCP environment to execute it. Would call: run_optimize(agent_module_path='src/agents/coordinator').", 'agent_module_path': 'src/agents/coordinator'}`

## 6 · Recap — from "seems to work" to measured & monitored  <sub>(slide 7)</sub>

You just walked the full **Quality Flywheel**:

**Design** metrics → **Execute** runs & traces → **Score** production with Online Monitors →
**Refine** with trace/BigQuery analysis + optimization → repeat.

### Run it live, end-to-end
```bash
# offline (what this notebook did — no credentials):
uv run python -m src.eval.demo.full_eval_demo --offline

# live against your deployed agent, and register the custom metrics:
gcloud auth application-default login
uv run python -m src.eval.demo.full_eval_demo --agent-id <AGENT_ENGINE_ID> --register-metrics \
    --emit-json eval_outputs/demo/full_demo.json
```

### Where to go next
- 📊 Slides: [`docs/eval_slides.pptx`](eval_slides.pptx) · [`eval_slides.html`](eval_slides.html)
- 📄 Operations reference + doc-coverage matrix: [`docs/eval_operations.md`](eval_operations.md)
- 📓 Guided walkthrough: [`docs/evaluation_demo.md`](evaluation_demo.md)
- 🧪 Code-runner notebook: [`src/eval/demo/evaluation_demo.ipynb`](../src/eval/demo/evaluation_demo.ipynb)

*Built clean on current Google clients from PyPI, with the eval suite passing.*